#  SETUP

In [0]:

%pip install xgboost mlflow scikit-learn matplotlib --quiet
dbutils.library.restartPython()


In [0]:
import seaborn as sns
import sys, os
notebook_path = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_path, ".."))
FOLDER_NAME = "src" 

In [0]:
src_path = os.path.join(project_root, FOLDER_NAME)
if not os.path.exists(src_path): src_path = os.path.join(notebook_path, FOLDER_NAME)
if src_path not in sys.path: sys.path.append(src_path)


In [0]:

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nombre_paquete.training import trainer 

## CARGAR DATOS SILVER

In [0]:
# Leemos las tablas que guardaste con nombres limpios
train_df = spark.table("climate_train_silver").toPandas()
test_df = spark.table("climate_test_silver").toPandas()

# Definimos Target
TARGET = 'energy_consumption'

# Columnas a excluir (Target, Fechas, Indices generados por Spark)
# 'index' suele aparecer cuando usamos reset_index() antes de guardar
cols_to_drop = [TARGET, 'index', 'date', 'level_0'] 

# Separar X e y
# Usamos una comprensión de lista para borrar solo si existen las columnas
X_train = train_df.drop(columns=[c for c in cols_to_drop if c in train_df.columns])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns])
y_test = test_df[TARGET]

print(f"Datos listos.")
print(f"Features ({X_train.shape[1]}): {X_train.columns.tolist()[:5]}...")

In [0]:
X_train.columns.tolist()

In [0]:

# --- CELDA 5: DEFINIR EXPERIMENTOS ---
# Lista de modelos a probar: (nombre, usar_busqueda_hiperparametros)
modelos_a_correr = [
    ("linear", False),       # Baseline rápido
    ("random_forest", True), # Modelo robusto
    ("xgboost", True)        # Modelo potente (Gradient Boosting)
]

In [0]:

for model_name, do_tuning in modelos_a_correr:
    
    # 1. Protección contra runs colgados
    if mlflow.active_run():
        print(" Se detectó un run activo. Cerrándolo...")
        mlflow.end_run()

    run_name = f"{model_name}_tuned" if do_tuning else model_name
    
    # 2. Iniciar Run
    with mlflow.start_run(run_name=run_name):
        print(f"\n Iniciando Run: {run_name}")
        
        # A. Entrenar
        model = trainer.train_model(X_train, y_train, model_type=model_name, tune_hyperparams=do_tuning)
        
        # B. Evaluar
        metrics, preds = trainer.evaluate_model(model, X_test, y_test)
        
        # C. Feature Importance (Opcional visualmente)
        print(f" Auditando variables para: {model_name}")
        feat_df = trainer.get_feature_importance(model, X_train.columns, model_type=model_name)
        
        if feat_df is not None:
            # Gráfica de importancia (Solo visualización)
            plt.figure(figsize=(10, 6))
            top_10 = feat_df.head(10)
            sns.barplot(data=top_10, x='abs_importance', y='feature', palette='viridis')
            plt.title(f'Top 10 Variables - {model_name}')
            plt.tight_layout()
            plt.show()
            plt.close()
            
            # Chequeo rápido de Leakage
            top_score = top_10.iloc[0]['abs_importance']
            total_score = top_10['abs_importance'].sum()
            if model_name != 'linear' and (top_score / total_score) > 0.85:
                 print(f" ALERTA DE LEAKAGE: '{top_10.iloc[0]['feature']}' domina el modelo.")

        # -----------------------------------------------------------
        # D. LOGUEAR A MLFLOW (AHORA ESTÁ FUERA DEL IF ANTERIOR)
        # -----------------------------------------------------------
        
        # 1. Parámetros
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("tuned", do_tuning)
        
        if do_tuning and hasattr(model, 'get_params'):
            params = model.get_params()
            # Loguea parámetros clave según el modelo
            if model_name == 'xgboost':
                mlflow.log_param("n_estimators", params.get('n_estimators'))
                mlflow.log_param("learning_rate", params.get('learning_rate'))
            elif model_name == 'random_forest':
                mlflow.log_param("n_estimators", params.get('n_estimators'))
                mlflow.log_param("max_depth", params.get('max_depth'))

        # 2. Métricas (Lo más importante)
        mlflow.log_metrics(metrics)
        print(f" Resultados {model_name}: RMSE={metrics['rmse']:.4f}, R2={metrics['r2']:.4f}")
        
        # 3. Guardar el Modelo (Artifact)
        if model_name == 'xgboost':
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")
            
        # 4. Gráfico de Predicciones
        fig, ax = plt.subplots(figsize=(8,6))
        ax.scatter(y_test, preds, alpha=0.5, label='Predicciones')
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfecto')
        ax.set_title(f'Predicciones: {run_name}')
        mlflow.log_figure(fig, "pred_vs_real.png")
        plt.close(fig)

print("\n Todos los experimentos terminados.")